# Time Series Data

In [12]:
!pip cache purge --quiet

In [19]:
import pandas as pd

In [20]:
# If you've forked this repo, change OWNER to your GitHub username.
# REPO and BRANCH will normally stay the same unless you renamed them.
OWNER = "singlestore-cookbook"
REPO = "singlestore-cookbook.github.io"
BRANCH = "refs/heads/main"

BASE_URL = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/code/part-multi-model/time-series-data/datasets"

In [21]:
tick_csv_url = f"{BASE_URL}/all_stocks_5yr.csv"

tick_df = pd.read_csv(tick_csv_url)

In [22]:
tick_df = tick_df.dropna()

In [23]:
tick_df.count()

date      619029
open      619029
high      619029
low       619029
close     619029
volume    619029
Name      619029
dtype: int64

In [24]:
tick_df = tick_df.drop(columns = ["open", "high", "low", "volume"])
tick_df = tick_df.rename(columns = {"date": "ts", "close": "price", "Name": "symbol"})
tick_df = tick_df.sort_values(by = ["ts", "symbol"])

In [25]:
tick_df.head()

,ts,price,symbol
71611,2013-02-08,45.0800,A
0,2013-02-08,14.7500,AAL
2518,2013-02-08,78.9000,AAP
1259,2013-02-08,67.8542,AAPL
3777,2013-02-08,36.2500,ABBV


<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [28]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [29]:
with db_connection.begin() as conn:
    conn.execute(text("TRUNCATE TABLE tick;"))

In [30]:
tick_df.to_sql(
    "tick",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

619029